In [1]:
import pandas as pd
from openpyxl.styles import Font, Alignment
import numpy as np

In [2]:
# FAZ O UPLOAD DA PLANILHA COM TODAS AS ABAS
abas = pd.read_excel(r"C:\Users\jeanl\Downloads\agregado_resultado_planilha_base_2016_2025_BR_CE_FORTALEZA_BA_NE_SP_MG (1).xlsx",
                         sheet_name=None)

In [ ]:
def trata(df):
    periodo = "PNADC"
    localidades = ['Brasil', 'Nordeste', 'Ceará']
    raca = [np.nan]
    sexo = [np.nan]
    faixa_etaria = [np.nan]
    nivel_instrucao = [np.nan]

    df_filtrado = df[
        (df["Fonte"].str.contains(periodo))&
        (df["Local"].isin(localidades)) &
        (df["Sexo"].isin(sexo)) &
        (df["Faixa etária"].isin(faixa_etaria)) &
        (df["Nível de instrução"].isin(nivel_instrucao)) &
        (df["Raça"].isin(raca))
         ]

    return df_filtrado

In [4]:
# FAZ ISSO PARA TODAS AS ABAS, MENOS A FOLHA DE ROSTO
abas_tratadas = {nome: trata(df) for nome, df in abas.items() if nome != "Sheet 1"}

In [5]:
#JUNTA TUDO EM UMA SÓ TABELA
df_ = pd.concat(
    [df.assign() for nome, df in abas_tratadas.items()],
    ignore_index=True
)

In [8]:
# RENOMEIA A FONTE

df_["Fonte"] = df_["Fonte"].replace({"PNADC_01":"1° Tri. ", 
                                    "PNADC_02":"2° Tri. ",
                                    "PNADC_03":"3° Tri. ",
                                    "PNADC_04":"4° Tri. "},
                                    regex = True)


In [6]:
df_.to_csv('dados_dash.csv')

In [ ]:
# TORNA A COLUNA FONTE O ÍNDICE
'''
df_.set_index("Fonte", inplace= True)
'''

'\ndf_final.set_index("Fonte", inplace= True)\n'

In [ ]:
# SE FOR MAIS DE UM DADO, CRIA UMA TABELA PIVOTADA

df_final = df_.pivot(index = 'Fonte',
columns = ['Raça', 'Local'],
values = ['Taxa de desocupação', 'Nível da ocupação', 'Participação'])

df_final = df_final.sort_index(axis=1, level=[0, 1]) # PRIMEIRO UM CATEGORIA E DEPOIS OS LOCAIS


In [ ]:
#TIRA O NOME DO ÍNDICE

df_final.columns.names = [None] * df_final.columns.nlevels
df_final.index.name = None

In [ ]:
#CRIA O ARQUIVO EXCEL
arquivo = 'terceiros_tris.xlsx'
with pd.ExcelWriter(arquivo, engine = "openpyxl") as writer:
    df_final.to_excel(
    writer,
    sheet_name = 'Dados',
    startrow = 1,
    index = True # FALSE SE A TABELA NÃO É PIVOTADA
)
# COLOCA TÍTULO, MEXER APENAS NO QUE VAI SER ESCRITO
    ws = writer.sheets['Dados']
    titulo = 'Brasil - Nível de ocupação, nível de participação e taxa de desocupação - 3° trimestres de 2016 a 2025'
    ws['A1'] = titulo
    ws.merge_cells(
    start_row = 1,
    start_column = 1,
    end_row = 1,
    end_column = len(df_final.columns) + 1
)
# CRIA A FONTE, MEXER APENAS NO QUE VAI SER ESCRITO
    ws['A1'].font = Font(bold = True, size = 10)
    ws['A1'].alignment = Alignment(horizontal = 'center')
    linha_fonte = df_final.shape[0] + 5
    ws[f'A{linha_fonte}'] = 'Fonte: IBGE - PNAD Contínua (microdados), janeiro de 2026. Elaboração: Observatório de Políticas Públicas do Trabalho do Estado do Ceará/SET.'
    ws.merge_cells(
    start_row = linha_fonte,
    start_column = 1,
    end_row = linha_fonte,
    end_column = len(df_final.columns) + 1
)
    ws[f"A{linha_fonte}"].font = Font(size=10)
    ws[f"A{linha_fonte}"].alignment = Alignment(horizontal="left")
